# 최종 정제 데이터셋 EDA 그래프 저장

`data/processed/최종정제_v2.csv`의 `final_text`를 기준으로 다음 그래프를 저장합니다.

- 최종 정제 댓글 별 글자 분포
- 최종 정제 댓글 별 토큰 수 분포
- 상위 20개 Bigram 빈도
- 상위 20개 단어 빈도


In [ ]:
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.util import ngrams

# 노트북을 루트에서 실행하든 notebooks 폴더에서 실행하든 경로가 맞도록 처리
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INPUT_PATH = ROOT / "data" / "processed" / "최종정제_v2.csv"
FIGURE_DIR = ROOT / "results" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# 폰트 설정 (Windows 기준 맑은 고딕)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="Malgun Gothic")

print("입력 파일:", INPUT_PATH)
print("그림 저장 폴더:", FIGURE_DIR)

In [ ]:
df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")
df["final_text"] = df["final_text"].fillna("").astype(str)
df["token_list"] = df["final_text"].apply(lambda text: text.split())
df["token_count"] = df["token_list"].apply(len)
df["char_count"] = df["final_text"].str.len()

all_tokens = [token for tokens in df["token_list"] for token in tokens]
word_counts = Counter(all_tokens)
bigram_counts = Counter(ngrams(all_tokens, 2))

print(f"전체 댓글 수: {len(df):,}")
print(f"전체 토큰 수: {len(all_tokens):,}")
print(f"고유 토큰 수: {len(word_counts):,}")
df[["final_text", "token_count", "char_count"]].head()

## 1. 최종 정제 댓글 별 글자 분포

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(df["char_count"], bins=50, kde=True, color="steelblue", ax=ax)
mean_chars = df["char_count"].mean()
median_chars = df["char_count"].median()
ax.axvline(mean_chars, color="crimson", linestyle="--", linewidth=1.5, label=f"평균: {mean_chars:.1f}")
ax.axvline(median_chars, color="darkorange", linestyle=":", linewidth=1.8, label=f"중앙값: {median_chars:.0f}")
ax.set_title("최종 정제 댓글 별 글자 수 분포", fontsize=15)
ax.set_xlabel("글자 수")
ax.set_ylabel("댓글 수")
ax.legend()
fig.tight_layout()

save_path = FIGURE_DIR / "eda_char_count_dist.png"
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print("저장:", save_path)

## 2. 최종 정제 댓글 별 토큰 수 분포

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(df["token_count"], bins=50, kde=True, color="skyblue", ax=ax)
mean_tokens = df["token_count"].mean()
median_tokens = df["token_count"].median()
ax.axvline(mean_tokens, color="crimson", linestyle="--", linewidth=1.5, label=f"평균: {mean_tokens:.1f}")
ax.axvline(median_tokens, color="darkorange", linestyle=":", linewidth=1.8, label=f"중앙값: {median_tokens:.0f}")
ax.set_title("최종 정제 댓글 별 토큰 수 분포", fontsize=15)
ax.set_xlabel("토큰 수")
ax.set_ylabel("댓글 수")
ax.legend()
fig.tight_layout()

save_path = FIGURE_DIR / "eda_token_count_dist.png"
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print("저장:", save_path)

## 3. 상위 20개 Bigram 빈도

In [ ]:
top20_bigrams = pd.DataFrame(
    [(" ".join(bigram), count) for bigram, count in bigram_counts.most_common(20)],
    columns=["bigram", "count"],
)

fig, ax = plt.subplots(figsize=(11, 8))
sns.barplot(data=top20_bigrams, x="count", y="bigram", color="mediumpurple", ax=ax)
ax.set_title("상위 20개 Bigram 빈도", fontsize=15)
ax.set_xlabel("빈도")
ax.set_ylabel("Bigram")
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3, fontsize=9)
fig.tight_layout()

save_path = FIGURE_DIR / "eda_top20_bigrams.png"
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print("저장:", save_path)
top20_bigrams

## 4. 상위 20개 단어 빈도

In [ ]:
top20_words = pd.DataFrame(word_counts.most_common(20), columns=["word", "count"])

fig, ax = plt.subplots(figsize=(11, 8))
sns.barplot(data=top20_words, x="count", y="word", color="seagreen", ax=ax)
ax.set_title("상위 20개 단어 빈도", fontsize=15)
ax.set_xlabel("빈도")
ax.set_ylabel("단어")
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3, fontsize=9)
fig.tight_layout()

save_path = FIGURE_DIR / "eda_top20_words.png"
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print("저장:", save_path)
top20_words